# HT5955 | Ingeniería de variables, HPO y entrega final

## Pregunta y alcance

¿Qué representaciones pequeñas de la historia del cliente ayudan a LightGBM a
ordenar mejor el riesgo de **BAJA+2**, manteniendo constante el modelo durante la
comparación de variables? No buscamos todas las combinaciones posibles ni el
mejor resultado de una semilla afortunada.

Este notebook es autocontenido: las funciones R están incluidas y documentadas
una por una. No necesita los scripts Python que lo construyeron. Hay cinco paquetes,
cinco semillas pareadas por paquete, una HPO posterior sobre columnas congeladas
y una etapa final de septiembre. **Los controles están preparados para la corrida
completa: abrir el archivo no entrena, pero ejecutar todas las celdas sí inicia el
estudio y permite las subidas autorizadas al finalizar.**

### Mapa de lectura

1. Revisar controles, definición del problema y calendario.
2. Entender datos, reparaciones, paquetes y regla de selección.
3. Leer las funciones por bloques: reproducibilidad, datos, ajustes, recursos y HPO.
4. Habilitar la ejecución explícitamente e inspeccionar todos los resultados.
5. Revisar la entrega y el procedimiento de recuperación.

**Limitación central:** mayo se utiliza para desarrollar variables y luego ajustar
hiperparámetros. Por lo tanto, no es un test independiente y su mejor AUC será
optimista por selección. Cinco semillas miden variación estocástica; no eliminan
sesgo de selección ni garantizan generalización. Septiembre es una población de
aplicación sin etiquetas disponibles, no una nueva partición de validación.


## 1. Controles de ejecución y seguridad

Usar un kernel **R / IRkernel**, desde el principio y en orden. Mantener el motor
nativo LightGBM instalado; no actualizarlo ni reemplazarlo desde este notebook.
Se necesita `DiceKriging` para el surrogate bayesiano, además de `data.table`,
`digest` y `jsonlite`. Si falta DiceKriging, `INSTALL_AUXILIARY_PACKAGES=TRUE`
permite instalarlo explícitamente. Nunca se reinstala LightGBM ni se actualizan
paquetes ya disponibles de forma automática.

`RUN_FULL=TRUE` y `FULL_RESOURCES_CONFIRMED=TRUE` ya están configurados por pedido
del usuario para la corrida completa. Se verificó la terminación del trabajador
residual del estudio anterior; antes de ejecutar no debe iniciarse otro estudio
que consuma esos mismos doce vCPUs.
`SAMPLE_MODE=TRUE` es una prueba sintética de cinco ajustes, sin HPO ni entregas.
El usuario autorizó las subidas y ambos controles, `SEND_KAGGLE` y
`COMPETITION_CONFIRMED`, están en TRUE. Al habilitar la corrida completa se
intentará subir los nueve archivos al terminar; abrir este notebook no los sube.
La competencia de referencia del directorio es
`data-mining-inicial-2026-c`; no se copia ciegamente el nombre de la plantilla z635.


In [ ]:
CLOUD_MODE <- TRUE
INSTALL_AUXILIARY_PACKAGES <- FALSE
SAMPLE_MODE <- FALSE
RUN_FULL <- TRUE
FULL_RESOURCES_CONFIRMED <- TRUE
WORKERS <- 3L
SEND_KAGGLE <- TRUE
COMPETITION_CONFIRMED <- TRUE
COMPETITION <- "data-mining-inicial-2026-c"
RESUME_DEAD_RUN_TOKEN <- ""
DEAD_WORKERS_CONFIRMED <- FALSE


In [ ]:
if (INSTALL_AUXILIARY_PACKAGES && !requireNamespace("DiceKriging", quietly = TRUE)) {
  install.packages("DiceKriging", repos = "https://cloud.r-project.org")
}
required <- c("data.table", "digest", "jsonlite", "lightgbm", "DiceKriging")
missing <- required[!vapply(required, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing)) stop("Faltan paquetes: ", paste(missing, collapse = ", "))
library(data.table)
data.table::setDTthreads(4L)
data.frame(paquete = required, version = vapply(required, function(p) as.character(packageVersion(p)), character(1)))


## 2. Calendario, población y objetivo

Cada fila es una fotografía mensual de un titular de paquete Premium. La salida
del paquete no implica necesariamente cerrar toda relación con el banco.
El objetivo binario es siempre `BAJA+2=1`; `BAJA+1` y `CONTINUA` valen 0.

| Etapa | Etiquetas para ajustar | Población aplicada | Uso de sus etiquetas |
| --- | --- | --- | --- |
| Ingeniería y confirmación | 201901–202103, 27 meses | 202105 | Selección y evaluación de desarrollo |
| HPO con receta congelada | 201901–202103 | 202105 | Optimizar AUC; no modificar columnas |
| Modelo final | 201901–202107, 31 meses | 202109 | Ninguna etiqueta de septiembre |

La fotografía t se publica el día 18 de t+1; las etiquetas completas admitidas
llegan hasta t−2. Para mayo pueden utilizarse covariables de abril y mayo, pero
no sus etiquetas en el ajuste. Para septiembre se permite historia de agosto y
septiembre y etiquetas hasta julio. No se usan desapariciones futuras del panel
como atributos ni se restringe entrenamiento a clientes que sobrevivan a mayo.

Se usan todas las filas elegibles, sin undersampling ni pesos temporales ocultos.
ID, mes y clase son claves o resultados, no predictores. Los montos, incluso los
que terminan en `_dolares`, están expresados en pesos equivalentes. Los estados de
tarjeta son códigos; sus diferencias numéricas no representan una distancia.


## 3. Historia existente y reparación de mediciones

El CSV trae 152 predictores, **ningún lag, delta o promedio explícito**.
`ctrx_quarter` ya cuenta movimientos de 90 días, `active_quarter` ya resume
actividad trimestral y `mrentabilidad_annual` ya acumula rentabilidad pasada;
ninguno es automáticamente una media mensual.

La fuente única es `analistasr_competencia_2026.csv`, tanto en `../` local como
en la ruta de datasets de la VM. No se incorpora otra base. La antigua receta
derivada de 278 columnas del proyecto se eliminó de esta corrida para ahorrar
cinco ajustes y todo su preprocesamiento adicional. El único baseline conserva
los 152 predictores originales, con las seis reparaciones autorizadas.

### Seis pares aprobados: cero de reporte se convierte en NA

| Campo | Mes afectado |
| --- | --- |
| mcuentas_saldo | 202006 |
| ctarjeta_debito_transacciones | 202006 |
| mtarjeta_visa_consumo | 202006 |
| mtarjeta_master_consumo | 202006 |
| chomebanking_transacciones | 202006 y 201910 |

En esas mediciones todos los clientes tienen cero. Un ejemplo: saldo 100000,
luego cero de reporte y luego 110000 produciría una caída y una recuperación
ficticias. Se marca desconocido **antes** de calcular historia en baseline,
candidatos, HPO y producción. Se preservan ceros normales, saldos negativos y el
CSV original. No se interpola desde meses futuros. El control se llama
`snapshot152_repaired`, no “CSV intacto”.
Los caches antiguos no reparados no se heredan como equivalentes.


## 4. Cinco hipótesis, no 86 candidatos

Alcance común: `mcuentas_saldo`, `mpayroll`, `ctrx_quarter`,
`ctarjeta_debito_transacciones`, `mtarjeta_visa_consumo`, `mtarjeta_master_consumo`.

| Paquete | Columnas nuevas | Definición | Hipótesis |
| --- | ---: | --- | --- |
| lag1_core | 6 | x[t−1] por mes calendario | Diferenciar nivel bajo habitual de una caída reciente |
| delta1_core | 6 | x[t]−x[t−1] | Exponer retirada de fondos o descenso de actividad |
| prior3_core | 12 | Media t−1:t−3, mínimo dos valores; más su conteo | Aproximar comportamiento usual sin depender de un solo mes |
| relative3_funds | 4 | Cambio relativo contra esa media y flag de escala casi cero, en saldo y nómina | Distinguir la misma caída absoluta entre clientes de escalas diferentes |
| card_utilization | 2 | Saldo/límite de compra para Master y Visa | Expresar presión relativa sobre capacidad disponible |

**Tratamiento exacto:** huecos de calendario o NA no son cero. Una media requiere
dos valores finitos y excluye el actual. El cambio relativo es
`clip((actual−media)/max(abs(media),1),−10,10)` y el flag es `abs(media)<1`.
La utilización exige denominador positivo y se recorta a [−10,10]. Los límites
son decisiones de robustez, no leyes financieras. La media del `ctrx_quarter`
suaviza ventanas de 90 días solapadas, no cuenta meses independientes.

Cada ajuste incorpora un paquete completo contra el mismo baseline, no encima
del ganador anterior. Lags/deltas/medias comparten alcance para comparar la forma
de representar historia. Los dos cambios relativos representan conjuntamente
la escala financiera personal; ambas utilizaciones cubren distintas carteras de
tarjetas. La atribución es al **paquete**, no a cada columna individual.

### Por qué LightGBM podría aprovecharlos

La historia agrega información que no existe en una fotografía aislada. Una
diferencia o razón es derivable si ya tenemos sus operandos, pero puede expresar
con pocos cortes una relación oblicua como “actividad menor a la mitad de lo
habitual”. LightGBM usa histogramas y crecimiento por hojas: su capacidad es
finita, y muchas variables correlacionadas compiten por cortes y muestreo.
Transformar monótonamente cada monto con log no agrega por sí solo orden nuevo.

Se posponen otros horizontes, EMA, tendencias, volatilidad, transiciones y
features móviles/transferencias con cambios de medición. No se amplía el registro
después de mirar qué candidato ganó en mayo.


## 5. Selección, HPO y presupuesto

La métrica principal confirmada es **AUC**. Un paquete se conserva si mejora la
AUC media pareada y gana en al menos cuatro de cinco semillas. Luego se prueba
una sola unión de paquetes seleccionados. Si esa unión falla la misma regla,
la corrida se detiene para revisión: no se buscan subconjuntos hasta ganar.
Si ninguno gana, la receta permanece en el baseline. Ganancia al 7%, logloss y
Brier se reportan sin sustituir el criterio de selección.

| Trabajo | Máximo de ajustes |
| --- | ---: |
| Un baseline × cinco semillas | 5 |
| Cinco paquetes × cinco semillas | 25 |
| Una confirmación × cinco semillas | 5 |
| HPO: veinte evaluaciones, una ya disponible | 19 |
| Entrenamiento final × cinco semillas | 5 |
| **Máximo de entrenamientos nuevos** | **59** |

El registro tiene 60 tareas lógicas; una evaluación HPO reutiliza obligatoriamente
la sonda ya ajustada, dejando como máximo 59 entrenamientos nuevos. Se ahorran
otros cinco si la confirmación coincide exactamente con un paquete terminado o
con el baseline. La HPO utiliza siempre la primera semilla: no cinco ajustes por configuración ni
una semilla diferente para cada propuesta. Se optimizan rondas 64–384, learning
rate 0.02–0.10, hojas 16–64, mínimo de filas 300–1500 y fracción de columnas 0.5–1.
El resto queda fijo. No hay early stopping con mayo ni modelos auxiliares ocultos.

Tres procesos de cuatro hilos aprovechan 12 vCPUs si hay RAM disponible. La HPO
usa un único proceso gaussiano y fantasías de media posterior (Kriging believer)
para proponer tres puntos antes de recibir sus resultados. Es menos adaptativa
que decidir tras cada ajuste, pero reduce la espera: aproximadamente siete ondas
de duración igual, no veinte, con tres puestos admitidos. El orden de finalización
no cambia la secuencia de propuestas.

Las **48 horas son un techo de planificación, no una ETA medida**: preparación 4,
comparaciones 16, HPO 20, producción/entrega 4 y recuperación 4. Primero se miden
controles contados y un punto HPO de costo alto. Se reserva el pico futuro de los
procesos activos y se usan menos puestos si hace falta. Si la proyección no cabe,
se pausa; no se reducen silenciosamente semillas, filas o rondas. Una interrupción
Spot o recursos compartidos pueden impedir cumplir el plazo.


## 6. Plan confirmado y semillas

La semilla principal es **333803**. Está en `z291_TareaHogar_02_senior.ipynb`,
`z293_Modelos_OOF_Ensembles.ipynb` y `o54_config()` de `z523_overnight_engine.R`.
La plantilla z635 usa 102191; no es nuestra referencia de proyecto. En z294 hay
una discrepancia entre texto (333803) y código (989831), advertida por el README.
Aquí no se intenta reproducir aquella partición: se firma un estudio nuevo con
cinco semillas derivadas de la principal, iguales para todos los candidatos.

El siguiente JSON es el registro legible y embebido. Cambiar un campo no permite
reutilizar el contrato previo. Las funciones siguientes son el código efectivo;
se conservan sus hashes y las versiones nativas al iniciar la corrida.


In [ ]:
plan <- jsonlite::fromJSON(r"---(
{
  "status": "user_confirmed_design_runtime_contract_created_on_launch",
  "supersedes": "86-candidate proposal in exp/HT5955/candidate_review_v2/proposed_candidate_registry.csv",
  "wall_budget_hours": 48,
  "target_positive": "BAJA+2",
  "master_seed": 333803,
  "master_seed_sources": ["z291_TareaHogar_02_senior.ipynb: PARAM$semilla_primigenia", "z293_Modelos_OOF_Ensembles.ipynb: PARAM$semilla", "z523_overnight_engine.R: o54_config seed"],
  "primary_baseline_proposal": "snapshot152_repaired",
  "core_fields": [
    "mcuentas_saldo", "mpayroll", "ctrx_quarter",
    "ctarjeta_debito_transacciones", "mtarjeta_visa_consumo", "mtarjeta_master_consumo"
  ],
  "candidates": [
    {"id": "lag1_core", "scope": "core_fields", "columns": 6, "formula": "x[t-1], exact calendar lookup"},
    {"id": "delta1_core", "scope": "core_fields", "columns": 6, "formula": "x[t]-x[t-1], both finite"},
    {"id": "prior3_core", "scope": "core_fields", "columns": 12, "formula": "prior-only mean t-1:t-3, minimum 2 finite values, plus known count"},
    {"id": "relative3_funds", "scope": ["mcuentas_saldo", "mpayroll"], "columns": 4, "formula": "clip((current-prior3_min2_mean)/max(abs(prior3_min2_mean),1),-10,10), plus near-zero prior flag"},
    {"id": "card_utilization", "scope": ["Master", "Visa"], "columns": 2, "formula": "clip(msaldototal/mlimitecompra,-10,10), finite numerator and positive finite denominator"}
  ],
  "feature_seeds_per_view": 5,
  "selection": {"metric": "auc", "mean_paired_delta_strictly_positive": true, "minimum_seeds_improved": 4, "aggregate_failure": "stop_for_review_no_subset_search"},
  "feature_stage": {"controls": 1, "candidate_views": 5, "aggregate_views": 1, "maximum_fits": 35},
  "hpo_proposal": {
    "method": "batch Bayesian optimization",
    "initial_trials": 8,
    "adaptive_trials": 12,
    "maximum_total_trials": 20,
    "maximum_new_fits": 19,
    "anchor_reuse_required": true,
    "anchor_trial": 2,
    "scheduling": {
      "maximum_parallel_trials": 3,
      "adaptive_batch_size": 3,
      "adaptive_batches": 4,
      "batch_proposal_policy": "Kriging-believer posterior-mean fantasies; propose three distinct configurations before dispatch",
      "update_policy": "replace fantasies with real results only after the complete batch is committed",
      "optimizer_writer": "controller_only",
      "resume_policy": "resume committed trial proposals and seeds; never redraw a pending batch",
      "initial_waves_at_three_slots": 3,
      "adaptive_waves_at_three_slots": 4,
      "total_equal_duration_waves_at_three_slots": 7,
      "total_equal_duration_waves_at_two_slots": 13,
      "total_equal_duration_waves_at_one_slot": 20
    },
    "seeds_per_trial": 1,
    "seed_policy": "FIT_SEEDS[1] for every trial; user approved",
    "metric": "auc",
    "search": {
      "num_iterations": {"lower": 64, "upper": 384, "type": "integer"},
      "learning_rate": {"lower": 0.02, "upper": 0.10, "type": "log_uniform"},
      "num_leaves": {"lower": 16, "upper": 64, "type": "integer"},
      "min_data_in_leaf": {"lower": 300, "upper": 1500, "type": "log_scaled_integer"},
      "feature_fraction": {"lower": 0.5, "upper": 1.0, "type": "uniform"}
    }
  },
  "final_fits": 5,
  "submission_cutoffs": [8000, 8500, 9000, 9500, 10000, 10500, 11000, 11500, 12000],
  "maximum_planned_fits": 60,
  "maximum_new_native_fits": 59,
  "time_envelopes_hours": {"preparation": 4, "feature_stage": 16, "hpo": 20, "final_and_delivery": 4, "recovery_reserve": 4},
  "native_threads_per_fit": 4,
  "maximum_parallel_fits": 3,
  "execution": {
    "feature_queue": "persistent PSOCK pool; refill on completion across candidates and seeds",
    "feature_precompute": "one exact calendar index; reusable label-free transformations per signed definition",
    "worker_inputs": "immutable local shard paths and task metadata; never export the full panel or controller sockets",
    "active_storage": "/content/ht5955_feature_development_scratch",
    "durable_storage": "new HT5955 bucket output; committed models and optimizer state required for recovery",
    "worker_data_table_threads": 1,
    "worker_blas_threads": 1,
    "bin_cache_policy": "key by matrix, training rows, max_bin, effective data_random_seed and other binning parameters",
    "bin_cache_lifecycle": "discard unique feature/final datasets after durable model commit; retain shared HPO dataset only through HPO; never delete durable models or other studies",
    "fresh_scratch_minimum_free_gib": 30,
    "runtime_scratch_reserve_gib": 8,
    "memory_policy": "reserve future peak allocations before dispatch; include controller and OS headroom",
    "resource_policy": "use only available capacity; preserve existing HT5954 jobs",
    "timing_policy": "measure counted controls under concurrency; forecast seven HPO waves only if three slots are admitted"
  },
  "recovery": {
    "durable_source_of_truth": "bucket; local scratch is reconstructible",
    "monthly_commit": "one SHA256 commit record per month, plus a final phase manifest",
    "model_commit": "native text and SHA256 metadata committed before predictions",
    "completed_task_recovery": "verify weights and return cached metrics without refitting",
    "model_only_recovery": "restore native model and finish prediction/metrics without refitting",
    "hpo_recovery": "reuse precommitted batch proposals and completed peer results; no redraw",
    "lock_takeover": "generation precondition plus explicit verified-dead controller/worker token",
    "uncommitted_fit": "retrain same task key and seed; no mid-tree continuation",
    "submission_retry": "never automatically retry a pending or ambiguous request"
  },
  "admission_required": true,
  "repair_policy": {
    "id": "ht5955_all_zero_to_na_v1",
    "approval": "user_approved",
    "action": "set listed field/month values to NA before any historical feature construction",
    "scope": "same repaired covariate source for the single snapshot baseline, all candidates, HPO and final training",
    "field_month_pairs": [
      {"field": "mcuentas_saldo", "month": 202006},
      {"field": "ctarjeta_debito_transacciones", "month": 202006},
      {"field": "mtarjeta_visa_consumo", "month": 202006},
      {"field": "mtarjeta_master_consumo", "month": 202006},
      {"field": "chomebanking_transacciones", "month": 202006},
      {"field": "chomebanking_transacciones", "month": 201910}
    ],
    "preserve_other_zeros": true,
    "modify_original_csv": false,
    "reuse_unrepaired_feature_caches": false,
    "applied_to_training_artifacts": false
  },
  "timing_measured": false
}
)---", simplifyVector = FALSE)


## 7. Reproducibilidad y definiciones básicas


### `h55_hash()`

Identidad de definiciones y tareas. Una huella distingue parámetros, columnas y semillas; un nombre parecido no demuestra equivalencia. Para el contrato y los modelos se verifica además el SHA256 de los bytes realmente guardados.


In [ ]:
h55_hash <- function(x) {
  digest::digest(jsonlite::toJSON(x, auto_unbox = TRUE, null = "null", digits = 17,
    dataframe = "rows", force = TRUE), algo = "sha256", serialize = FALSE)
}


### `h55_save()`

Escritura de checkpoints. Primero se termina un archivo temporal y después se publica el nombre definitivo. Los lectores ignoran temporales; nunca se considera entrenado un modelo que solo existe en memoria. No se utiliza este mecanismo para reemplazar contratos incompatibles.


In [ ]:
h55_save <- function(x, path) {
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  tmp <- tempfile(pattern = ".uncommitted_", tmpdir = dirname(path))
  on.exit(unlink(tmp), add = TRUE)
  saveRDS(x, tmp, compress = FALSE, version = 3L)
  if (!file.rename(tmp, path)) stop("Cannot commit artifact: ", path)
  invisible(path)
}


### `h55_read()`

Lectura de un checkpoint terminado. La ausencia devuelve NULL; un archivo corrupto produce error, no se interpreta como una autorización silenciosa para reentrenar o cambiar el estudio.


In [ ]:
h55_read <- function(path) {
  if (file.exists(path)) readRDS(path) else NULL
}


### `h55_rng()`

Aleatoriedad aislada. Guardamos y restauramos tanto RNGkind como el estado del generador del usuario. Derivar semillas o proponer hiperparámetros no debe consumir accidentalmente la secuencia de otro proceso experimental.


In [ ]:
h55_rng <- function(seed, fun) {
  kind <- RNGkind(); existed <- exists(".Random.seed", envir = .GlobalEnv)
  old <- if (existed) get(".Random.seed", envir = .GlobalEnv) else NULL
  on.exit({
    do.call(RNGkind, as.list(kind))
    if (existed) assign(".Random.seed", old, envir = .GlobalEnv)
    else if (exists(".Random.seed", envir = .GlobalEnv)) rm(".Random.seed", envir = .GlobalEnv)
  })
  set.seed(seed, kind = "Mersenne-Twister", normal.kind = "Inversion", sample.kind = "Rejection")
  fun()
}


### `h55_seeds()`

Cinco semillas compartidas, no cinco meses de validación. Partimos de 333803 y extraemos cinco enteros distintos una sola vez. El contrato congela sus valores y orden; ningún candidato, trabajador o reintento vuelve a sortearlos.


In [ ]:
h55_seeds <- function(master = 333803L) {
  h55_rng(master, function() sample.int(.Machine$integer.max, 5L, replace = FALSE, useHash = TRUE))
}


### `h55_config()`

Rutas absolutas y controles operativos. La fuente, la salida durable y el scratch se separan para que un cambio de directorio no cambie el experimento. El límite es tres ajustes simultáneos de cuatro hilos; Kaggle y la confirmación de recursos están desactivados inicialmente.


In [ ]:
h55_config <- function(cloud = TRUE, smoke = FALSE) {
  list(cloud = cloud, smoke = smoke,
    input = if (cloud) "/content/buckets/b1/datasets/analistasr_competencia_2026.csv" else normalizePath("../analistasr_competencia_2026.csv", mustWork = FALSE),
    output = if (cloud) "/content/buckets/b1/exp/HT5955/feature_development_v1" else "exp/HT5955/local_full_v1",
    scratch = if (cloud) "/content/ht5955_feature_development_scratch" else "/tmp/opencode/ht5955_local_scratch",
    bucket_root = "/content/buckets/b1", bucket_uri = "gs://cfleuryrangelteixei_buckito2026",
    expected_sha = "a97d955b59917ea0df831264e5b50dc2c407580e26adb08d2cbc798c32b199aa",
    master = 333803L, threads = 4L, workers = 3L, available_vcpus = 12L,
    full_resources_confirmed = FALSE, reserve_gib = 24, disk_min_gib = 30, disk_resume_min_gib = 8,
    resume_dead_token = "", dead_workers_confirmed = FALSE,
    send_kaggle = FALSE, competition_confirmed = FALSE,
    competition = "data-mining-inicial-2026-c", cutoffs = seq(8000L, 12000L, 500L),
    horizon_hours = 48, recovery_hours = 4, final_hours = 4)
}


## 8. Datos, calendario y banco causal


### `h55_core()`

Los seis campos con historia. Cubren fondos, nómina, actividad voluntaria, débito y consumo de ambas tarjetas. No se suman canales que podrían contar la misma transacción. El alcance es idéntico en lags, deltas y promedios para comparar representaciones.


In [ ]:
h55_core <- function() c("mcuentas_saldo", "mpayroll", "ctrx_quarter",
  "ctarjeta_debito_transacciones", "mtarjeta_visa_consumo", "mtarjeta_master_consumo")


### `h55_fixture()`

Panel sintético pequeño para la prueba técnica. Mantiene los 152 nombres de entrada, incluye un hueco de calendario y reproduce las seis reparaciones autorizadas. Sus etiquetas son artificiales: ninguna AUC o ganancia de este panel es evidencia económica.


In [ ]:
h55_fixture <- function(raw) {
  stopifnot(length(raw) == 152L)
  h55_rng(9001L, function() {
    d <- data.table::CJ(numero_de_cliente = 1:40, foto_mes = h55_months(202109L))
    for (field in raw) data.table::set(d, j = field, value = runif(nrow(d), 1, 100))
    for (brand in c("Master", "Visa")) d[, (paste0(brand, "_status")) := 0]
    d[, clase_ternaria := ifelse(numero_de_cliente %% 7L == 0L, "BAJA+2",
      ifelse(numero_de_cliente %% 11L == 0L, "BAJA+1", "CONTINUA"))]
    repairs <- h55_repairs()
    for (i in seq_len(nrow(repairs)))
      data.table::set(d, i = which(d$foto_mes == repairs$month[i]), j = repairs$field[i], value = 0)
    d <- d[!(numero_de_cliente == 1L & foto_mes == 202104L)]
    d
  })
}


### `h55_repairs()`

Registro cerrado de seis pares campo/mes aprobados. Es una decisión de calidad de medición, no una transformación general de ceros. No adoptamos automáticamente la lista mucho mayor de reparaciones del notebook del curso.


In [ ]:
h55_repairs <- function() data.table::data.table(
  field = c("mcuentas_saldo", "ctarjeta_debito_transacciones", "mtarjeta_visa_consumo",
    "mtarjeta_master_consumo", "chomebanking_transacciones", "chomebanking_transacciones"),
  month = c(rep(202006L, 5L), 201910L))


### `h55_repair()`

Aplicación de cero a NA antes de generar historia. Se comprueba el patrón de origen y se trabaja sobre una copia. Los ceros normales, los saldos negativos legítimos y el CSV original no se modifican. NA significa desconocido, no inactividad.


In [ ]:
h55_repair <- function(d, verify = TRUE) {
  out <- data.table::copy(d)
  registry <- h55_repairs()
  for (i in seq_len(nrow(registry))) {
    field <- registry$field[i]; month <- registry$month[i]
    stopifnot(field %in% names(out))
    at <- which(out$foto_mes == month)
    if (verify && length(at) && any(!is.finite(out[[field]][at]) | out[[field]][at] != 0))
      stop("Repair source mismatch: ", field, " month ", month)
    if (length(at)) data.table::set(out, i = at, j = field, value = NA)
  }
  out
}


### `h55_month()`

Calendario real. YYYYMM no es un entero al que podamos restar uno en enero. La conversión produce un índice mensual consecutivo; así diciembre y enero quedan separados por un mes y los huecos no se comprimen.


In [ ]:
h55_month <- function(x) {
  stopifnot(all(x %% 100L %in% 1:12))
  (x %/% 100L) * 12L + x %% 100L - 1L
}


### `h55_months()`

Meses de entrenamiento explícitos. Desarrollo utiliza los 27 meses de enero de 2019 a marzo de 2021; producción amplía hasta julio de 2021. Nunca seleccionamos clientes según su supervivencia en mayo o septiembre.


In [ ]:
h55_months <- function(last) {
  x <- seq.int(h55_month(201901L), h55_month(last))
  (x %/% 12L) * 100L + x %% 12L + 1L
}


### `h55_features()`

Banco causal compacto. Solo recibe covariables, nunca la clase. Las búsquedas son por cliente y mes exacto; las medias excluyen el mes actual. Construye únicamente las treinta columnas de los cinco paquetes y consulta hasta tres meses previos. No calcula el banco histórico de 278 columnas ni transformaciones que no se van a probar.


In [ ]:
h55_features <- function(panel, rows, raw, index = NULL) {
  stopifnot(!"clase_ternaria" %in% names(panel))
  if (is.null(index)) {
    stopifnot(!anyDuplicated(panel, by = c("numero_de_cliente", "foto_mes")))
    index <- data.table::data.table(id = panel$numero_de_cliente, month = h55_month(panel$foto_mes), row = seq_len(nrow(panel)))
    data.table::setkey(index, id, month)
  }
  current <- panel[rows]
  previous <- lapply(1:3, function(k) index[data.table::data.table(
    id = current$numero_de_cliente, month = h55_month(current$foto_mes) - k), on = .(id, month), row])
  out <- data.table::copy(current[, c("numero_de_cliente", "foto_mes", raw), with = FALSE])
  for (field in h55_core()) {
    v <- vapply(previous, function(at) as.double(panel[[field]][at]), numeric(nrow(current)))
    v[!is.finite(v)] <- NA_real_
    out[, (paste0(field, "_lag1")) := v[, 1L]]
    out[, (paste0(field, "_delta1")) := as.double(current[[field]]) - v[, 1L]]
    known <- rowSums(is.finite(v)); avg <- rowMeans(v, na.rm = TRUE); avg[known < 2L] <- NA_real_
    out[, (paste0(field, "_prior3_observed")) := known]
    out[, (paste0(field, "__prior3_min2_mean")) := avg]
    if (field %in% c("mcuentas_saldo", "mpayroll")) {
      out[, (paste0(field, "__relative_prior3_min2")) := pmax(-10, pmin(10, (as.double(current[[field]]) - avg) / pmax(abs(avg), 1)))]
      out[, (paste0(field, "__near_zero_prior3_min2")) := ifelse(is.na(avg), NA_integer_, as.integer(abs(avg) < 1))]
    }
  }
  for (brand in c("Master", "Visa")) {
    a <- current[[paste0(brand, "_msaldototal")]]; b <- current[[paste0(brand, "_mlimitecompra")]]
    good <- is.finite(a) & is.finite(b) & b > 0; value <- rep(NA_real_, length(a))
    value[good] <- pmax(-10, pmin(10, a[good] / b[good]))
    out[, (paste0(brand, "__utilization")) := value]
  }
  predictors <- setdiff(names(out), c("numero_de_cliente", "foto_mes"))
  for (field in predictors) {
    bad <- which(!is.finite(out[[field]]))
    if (length(bad)) data.table::set(out, i = bad, j = field, value = NA)
  }
  out
}


### `h55_views()`

Vistas de columnas, no combinaciones exhaustivas. Cada candidato es el mismo baseline de 152 predictores más un paquete y se prueba independientemente. La unión seleccionada comparte cálculos intermedios y no duplica definiciones idénticas. No hay un segundo baseline histórico.


In [ ]:
h55_views <- function(raw) {
  additions <- list(lag1_core = paste0(h55_core(), "_lag1"), delta1_core = paste0(h55_core(), "_delta1"),
    prior3_core = unlist(lapply(h55_core(), function(f) c(paste0(f, "__prior3_min2_mean"), paste0(f, "_prior3_observed")))),
    relative3_funds = unlist(lapply(c("mcuentas_saldo", "mpayroll"), function(f)
      paste0(f, c("__relative_prior3_min2", "__near_zero_prior3_min2")))),
    card_utilization = paste0(c("Master", "Visa"), "__utilization"))
  c(list(snapshot152_repaired = raw),
    lapply(additions, function(x) c(raw, x)))
}


## 9. Métricas y sonda fija


### `h55_metrics()`

Medición fuera del ajuste. AUC con empates promediados es el criterio principal confirmado. Ganancia al 7%, logloss y Brier son diagnósticos secundarios. El contacto fijo usa probabilidad descendente y desempate por ID; el ID no entra al modelo. Los productos de conteos se calculan en doble precisión para evitar overflow.


In [ ]:
h55_metrics <- function(prob, label, id) {
  stopifnot(length(prob) == length(label), !anyNA(label), all(label %in% c(0L, 1L)), all(is.finite(prob)))
  positives <- sum(as.double(label)); negatives <- length(label) - positives
  if (!positives || !negatives) stop("AUC needs both classes")
  auc <- (sum(rank(prob, ties.method = "average")[label == 1L]) - positives * (positives + 1) / 2) / (positives * negatives)
  at <- order(-prob, id); contacts <- floor(.07 * length(label)); tp <- sum(label[at[seq_len(contacts)]])
  p <- pmax(1e-15, pmin(1 - 1e-15, prob))
  list(auc = auc, gain7 = 1e6 * tp - 25000 * contacts, contacts = contacts, tp = tp,
    logloss = -mean(label * log(p) + (1 - label) * log1p(-p)), brier = mean((prob - label)^2))
}


### `h55_params()`

Sonda LightGBM fija durante ingeniería. Objetivo binario exclusivamente BAJA+2, 192 rondas y parámetros de referencia. Todas las fuentes de aleatoriedad nativa reciben la misma semilla de la réplica. El modo sintético baja explícitamente tamaño y rondas y no puede generar una entrega real.


In [ ]:
h55_params <- function(seed, overrides = list(), smoke = FALSE) {
  p <- list(objective = "binary", metric = "None", learning_rate = .04666, num_leaves = 52L,
    min_data_in_leaf = 443L, feature_fraction = .545, bagging_fraction = .786, bagging_freq = 1L,
    lambda_l2 = 8.7465, max_bin = 31L, num_iterations = 192L, num_threads = 4L,
    deterministic = TRUE, force_col_wise = TRUE, feature_pre_filter = FALSE,
    use_missing = TRUE, zero_as_missing = FALSE, verbosity = -1L,
    seed = seed, feature_fraction_seed = seed, bagging_seed = seed, data_random_seed = seed)
  if (smoke) p <- utils::modifyList(p, list(num_iterations = 5L, num_leaves = 7L, min_data_in_leaf = 8L))
  utils::modifyList(p, overrides)
}


### `h55_log()`

Trazabilidad de fases. Cada tarea tiene su propio archivo para evitar que trabajadores escriban sobre el mismo log. Se distinguen construcción del dataset, entrenamiento nativo, commit del modelo, predicción y métricas. Un kernel vivo no demuestra que esté entrenando.


In [ ]:
h55_log <- function(e, phase, message, key = "controller") {
  path <- file.path(e$cfg$output, "logs", paste0(key, ".log"))
  dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
  cat(format(Sys.time(), "%Y-%m-%dT%H:%M:%OS3Z", tz = "UTC"), phase, message, "\n", file = path, append = TRUE)
  if (key == "controller") message(phase, ": ", message)
}


## 10. Contratos, bloqueos y preparación durable


### `h55_storage()`

Operaciones de almacenamiento acotadas. Solo se usa gcloud para objetos del estudio y sus bloqueos; no se inician ni modifican VMs. Un fallo de autenticación o de red detiene la operación y no se disfraza de cache inexistente.


In [ ]:
h55_storage <- function(args, allow_failure = FALSE) {
  text <- suppressWarnings(system2("gcloud", vapply(c("storage", args), shQuote, character(1)), stdout = TRUE, stderr = TRUE))
  status <- attr(text, "status"); if (is.null(status)) status <- 0L
  if (status && !allow_failure) stop("GCS operation failed: ", paste(text, collapse = " | "))
  list(status = status, text = text)
}


### `h55_lock()`

Un controlador por estudio. En GCS la exclusión se obtiene con precondiciones de generación, no con un directorio de GCSFUSE. Recuperar un bloqueo exige el token anterior y confirmar que murieron controlador y trabajadores. La antigüedad del bloqueo por sí sola nunca autoriza robarlo.


In [ ]:
h55_lock <- function(cfg) {
  owner <- list(pid = Sys.getpid(), host = Sys.info()[["nodename"]], started = as.character(Sys.time()),
    token = h55_hash(list(Sys.getpid(), as.character(Sys.time()), tempfile())))
  local <- file.path(cfg$output, "RUNNING.lock")
  if (cfg$cloud && !cfg$smoke) {
    stopifnot(startsWith(cfg$output, paste0(cfg$bucket_root, "/")))
    uri <- paste0(cfg$bucket_uri, "/", substring(cfg$output, nchar(cfg$bucket_root) + 2L), "/RUNNING.json")
    tmp <- tempfile(tmpdir = cfg$scratch, fileext = ".json"); on.exit(unlink(tmp), add = TRUE)
    jsonlite::write_json(owner, tmp, auto_unbox = TRUE)
    claim <- h55_storage(c("cp", "--if-generation-match=0", tmp, uri), TRUE)
    if (claim$status) {
      generation <- h55_storage(c("objects", "describe", uri, "--format=value(generation)"))$text
      stopifnot(length(generation) == 1L, grepl("^[0-9]+$", generation))
      oldpath <- tempfile(tmpdir = cfg$scratch); on.exit(unlink(oldpath), add = TRUE)
      h55_storage(c("cp", paste0(uri, "#", generation), oldpath))
      old <- jsonlite::read_json(oldpath, simplifyVector = TRUE)
      if (!cfg$dead_workers_confirmed || !identical(cfg$resume_dead_token, old$token))
        stop("Run locked by ", old$host, " pid ", old$pid, ". Verify controller AND workers dead before resuming with token ", old$token)
      h55_storage(c("cp", oldpath, paste0(dirname(uri), "/run_history/", old$token, ".json")))
      h55_storage(c("rm", paste0("--if-generation-match=", generation), uri))
      h55_storage(c("cp", "--if-generation-match=0", tmp, uri))
    }
    gen <- h55_storage(c("objects", "describe", uri, "--format=value(generation)"))$text
    return(list(uri = uri, generation = gen, owner = owner))
  }
  if (!dir.create(local, showWarnings = FALSE)) {
    old <- h55_read(file.path(local, "owner.rds"))
    if (is.null(old) || !cfg$dead_workers_confirmed || !identical(old$token, cfg$resume_dead_token))
      stop("Local lock exists; verify controller/workers dead: ", local)
    if (identical(old$host, Sys.info()[["nodename"]]) && dir.exists(paste0("/proc/", old$pid))) stop("Old controller still exists")
    unlink(local, recursive = TRUE)
    if (!dir.create(local)) stop("Local lock takeover failed")
  }
  h55_save(owner, file.path(local, "owner.rds")); list(path = local, owner = owner)
}


### `h55_unlock()`

Liberación del bloqueo propio. Se utiliza la generación adquirida por este proceso para que un controlador viejo no pueda borrar el bloqueo de uno nuevo. Los trabajadores deben haberse detenido antes de liberar el estudio.


In [ ]:
h55_unlock <- function(lock) {
  if (!is.null(lock$uri)) h55_storage(c("rm", paste0("--if-generation-match=", lock$generation), lock$uri))
  else {
    old <- h55_read(file.path(lock$path, "owner.rds"))
    if (!is.null(old) && identical(old$token, lock$owner$token)) unlink(lock$path, recursive = TRUE)
  }
}


### `h55_scratch_path()`

Scratch válido aunque /content sea un enlace simbólico. Se resuelven tanto la raíz como el directorio de trabajo y se comprueba que este sea un descendiente real. En la VM actual /content apunta a /home/ds: no es un error ni obliga a mover datos. El chequeo de disco rechaza por separado tmpfs y GCSFUSE.


In [ ]:
h55_scratch_path <- function(scratch, root = "/content") {
  resolved_root <- normalizePath(root, mustWork = TRUE)
  resolved_scratch <- normalizePath(scratch, mustWork = TRUE)
  if (!startsWith(resolved_scratch, paste0(sub("/$", "", resolved_root), "/")))
    stop("Cloud scratch must be a dedicated directory under the resolved /content root")
  invisible(resolved_scratch)
}


### `h55_validate_plan()`

Coherencia entre el plan escrito y el motor. Antes de iniciar se comprueban paquetes, AUC, semillas, reparaciones, cortes y presupuesto de veinte ensayos HPO. Editar solo el JSON no puede hacer que el notebook prometa menos ajustes mientras el código ejecuta más; una modificación exige revisar también la implementación y crear otro contrato.


In [ ]:
h55_validate_plan <- function(plan, cfg) {
  ids <- vapply(plan$candidates, `[[`, character(1), "id")
  checks <- c(identical(ids, c("lag1_core", "delta1_core", "prior3_core", "relative3_funds", "card_utilization")),
    identical(as.integer(vapply(plan$candidates, `[[`, numeric(1), "columns")), c(6L, 6L, 12L, 4L, 2L)),
    identical(plan$target_positive, "BAJA+2"), identical(plan$selection$metric, "auc"),
    identical(plan$hpo_proposal$metric, "auc"), plan$selection$minimum_seeds_improved == 4L,
    plan$feature_seeds_per_view == 5L, plan$feature_stage$controls == 1L, plan$feature_stage$maximum_fits == 35L,
    is.null(plan$secondary_control), plan$hpo_proposal$anchor_reuse_required,
    plan$hpo_proposal$maximum_new_fits == 19L,
    plan$hpo_proposal$initial_trials == 8L, plan$hpo_proposal$adaptive_trials == 12L,
    plan$hpo_proposal$maximum_total_trials == 20L, plan$hpo_proposal$seeds_per_trial == 1L,
    plan$maximum_planned_fits == 60L, plan$maximum_new_native_fits == 59L,
    plan$final_fits == 5L, plan$master_seed == 333803L,
    identical(plan$repair_policy$approval, "user_approved"),
    identical(plan$repair_policy$id, "ht5955_all_zero_to_na_v1"),
    identical(unlist(plan$core_fields, use.names = FALSE), h55_core()),
    identical(as.integer(unlist(plan$submission_cutoffs)), cfg$cutoffs))
  repairs <- data.table::rbindlist(plan$repair_policy$field_month_pairs)
  expected <- h55_repairs()
  checks <- c(checks, nrow(repairs) == 6L,
    setequal(paste(repairs$field, repairs$month), paste(expected$field, expected$month)))
  bounds <- list(num_iterations = c(64, 384), learning_rate = c(.02, .10), num_leaves = c(16, 64),
    min_data_in_leaf = c(300, 1500), feature_fraction = c(.5, 1))
  for (name in names(bounds)) checks <- c(checks, isTRUE(all.equal(
    as.numeric(unlist(plan$hpo_proposal$search[[name]][c("lower", "upper")])), bounds[[name]])))
  if (!all(checks)) stop("Unsupported design: plan does not match this engine; use a new reviewed implementation/study")
  invisible(TRUE)
}


### `h55_setup()`

Contrato inmutable. Se verifican fuente, versiones, biblioteca nativa, código, plan, semilla principal y cinco semillas derivadas. El contrato se compara estrictamente y su firma proviene del archivo guardado. Cambiar cualquiera de esos elementos exige otro estudio, no borrar el contrato aceptado.


In [ ]:
h55_setup <- function(cfg, plan, source_sha = NULL) {
  launch_started <- as.numeric(Sys.time())
  h55_validate_plan(plan, cfg)
  packages <- c("data.table", "digest", "jsonlite", "lightgbm", "DiceKriging")
  for (p in packages) if (!requireNamespace(p, quietly = TRUE)) stop("Missing package: ", p, "; no native auto-upgrade is allowed")
  stopifnot(cfg$threads == 4L, cfg$workers %in% 1:3, cfg$workers * cfg$threads <= cfg$available_vcpus,
    cfg$master == plan$master_seed, cfg$horizon_hours == plan$wall_budget_hours)
  if (!cfg$smoke && parallel::detectCores(logical = TRUE) < cfg$workers * cfg$threads)
    stop("Requested native threads exceed detected CPUs")
  if (!cfg$smoke && (!cfg$full_resources_confirmed || !cfg$cloud))
    stop("Full fits require explicit confirmation of available cloud CPU/RAM; local mode is smoke-only")
  dir.create(cfg$scratch, recursive = TRUE, showWarnings = FALSE)
  dir.create(cfg$output, recursive = TRUE, showWarnings = FALSE)
  if (cfg$cloud) h55_scratch_path(cfg$scratch)
  h55_disk(cfg)
  lock <- h55_lock(cfg); success <- FALSE
  on.exit(if (!success) h55_unlock(lock), add = TRUE)
  if (is.null(source_sha)) source_sha <- digest::digest(file = cfg$input, algo = "sha256")
  if (!cfg$smoke && !identical(source_sha, cfg$expected_sha)) stop("Source data hash mismatch")
  functions <- sort(ls(envir = .GlobalEnv, pattern = "^h55_"))
  functions <- functions[vapply(mget(functions, .GlobalEnv), is.function, logical(1))]
  code <- lapply(functions, function(n) paste(deparse(get(n, .GlobalEnv), width.cutoff = 500L), collapse = "\n"))
  names(code) <- functions
  seeds <- h55_seeds(cfg$master)
  contract <- list(version = "HT5955_compact_v1", source_sha = source_sha, smoke = cfg$smoke,
    plan = plan, seeds = seeds, master = cfg$master, derivation = "sample.int_unique5_v1",
    rng = c("Mersenne-Twister", "Inversion", "Rejection"), R = R.version.string,
    platform = R.version$platform, external_libraries = as.list(extSoftVersion()),
    BLAS = sessionInfo()$BLAS, LAPACK = sessionInfo()$LAPACK,
    packages = setNames(lapply(packages, function(p) as.character(utils::packageVersion(p))), packages),
    native_sha = digest::digest(file = getLoadedDLLs()[["lightgbm"]][["path"]], algo = "sha256"),
    code = code, cutoffs = cfg$cutoffs, competition = cfg$competition)
  text <- as.character(jsonlite::toJSON(contract, auto_unbox = TRUE, null = "null", digits = 17, force = TRUE))
  path <- file.path(cfg$output, "contract.rds")
  old <- h55_read(path)
  if (!is.null(old) && !identical(old, text)) stop("Signed study mismatch: preserve this output and use a new study namespace")
  if (is.null(old)) h55_save(text, path)
  e <- new.env(parent = globalenv()); e$cfg <- cfg; e$plan <- plan; e$seeds <- seeds
  e$signature <- digest::digest(file = path, algo = "sha256")
  clock <- file.path(cfg$output, "clock.rds")
  if (!file.exists(clock)) h55_save(list(started = launch_started), clock)
  e$started <- h55_read(clock)$started; e$cluster <- NULL; e$pids <- integer(); e$lock <- lock
  e$ledger <- h55_read(file.path(cfg$output, "jobs.rds")); if (is.null(e$ledger)) e$ledger <- list()
  e$admission_cap <- cfg$workers
  success <- TRUE; e
}


### `h55_prepare()`

Preparación y staging antes de abrir el pool. Cada mes tiene su archivo y un registro de commit con SHA256; una preempción antes de completar el manifiesto global no pierde los meses ya confirmados. Se separan etiquetas, se aplican reparaciones y se construye un solo índice. En producción se reutilizan los meses idénticos de desarrollo; el staging local se reconstruye desde el bucket sin recalcularlos.


In [ ]:
h55_prepare <- function(e, phase = "development", fixture = NULL) {
  h55_disk(e$cfg)
  train <- h55_months(if (phase == "final") 202107L else 202103L)
  evaluation <- if (phase == "final") 202109L else 202105L
  months <- c(train, evaluation)
  root <- file.path(e$cfg$output, "bank", phase); local <- file.path(e$cfg$scratch, e$signature, "bank", phase)
  dir.create(local, recursive = TRUE, showWarnings = FALSE)
  manifest <- h55_read(file.path(root, "manifest.rds"))
  if (is.null(manifest)) {
    h55_log(e, "feature_preparation", phase)
    entries <- list(); raw <- NULL
    for (month in months) {
      committed <- h55_read(file.path(root, paste0(month, ".commit.rds")))
      if (!is.null(committed)) {
        stopifnot(identical(committed$signature, e$signature), committed$month == month,
          identical(digest::digest(file = file.path(root, committed$entry$file), algo = "sha256"), committed$entry$sha))
        entries[[as.character(month)]] <- committed$entry
        if (!is.null(raw)) stopifnot(identical(raw, committed$raw))
        raw <- committed$raw
      }
    }
    if (phase == "final") {
      prior <- h55_read(file.path(e$cfg$output, "bank", "development", "manifest.rds"))
      if (!is.null(prior)) {
        stopifnot(identical(prior$signature, e$signature))
        raw <- prior$raw
        for (m in intersect(as.character(months), names(prior$entries))) {
          entry <- prior$entries[[m]]
          entry$durable_path <- file.path(e$cfg$output, "bank", "development", entry$file)
          entries[[m]] <- entry
        }
      }
    }
    needed <- setdiff(months, as.integer(names(entries)))
    if (length(needed)) {
      data.table::setDTthreads(4L)
      if (is.null(fixture)) {
        staged <- file.path(e$cfg$scratch, "source.csv")
        source_sha <- jsonlite::fromJSON(h55_read(file.path(e$cfg$output, "contract.rds")))$source_sha
        if (!file.exists(staged) || !identical(digest::digest(file = staged, algo = "sha256"), source_sha)) {
          if (!file.copy(e$cfg$input, staged, overwrite = TRUE)) stop("Source staging failed")
          stopifnot(identical(digest::digest(file = staged, algo = "sha256"), source_sha))
        }
        d <- data.table::fread(staged, showProgress = TRUE)
      } else d <- data.table::copy(fixture)
      d <- d[foto_mes <= evaluation]
      stopifnot(!anyDuplicated(d, by = c("numero_de_cliente", "foto_mes")))
      data.table::setorder(d, numero_de_cliente, foto_mes)
      raw <- setdiff(names(d), c("numero_de_cliente", "foto_mes", "clase_ternaria"))
      stopifnot(length(raw) == 152L, all(vapply(d[, raw, with = FALSE], is.numeric, logical(1))))
    if (phase == "final") d <- d[h55_month(foto_mes) >= min(h55_month(needed)) - 3L]
      labels <- d[, .(numero_de_cliente, foto_mes, clase_ternaria)]
      d[, clase_ternaria := NULL]
      repaired <- h55_repair(d, verify = TRUE); rm(d)
      index <- data.table::data.table(id = repaired$numero_de_cliente,
        month = h55_month(repaired$foto_mes), row = seq_len(nrow(repaired)))
      data.table::setkey(index, id, month)
      for (month in needed) {
        at <- which(repaired$foto_mes == month)
        if (!length(at)) stop("Missing required month: ", month)
        frame <- h55_features(repaired, at, raw, index)
        target <- labels$clase_ternaria[at]
        if (phase != "final" || month != evaluation) {
          if (anyNA(target) || !all(target %in% c("CONTINUA", "BAJA+1", "BAJA+2"))) stop("Unknown eligible labels at ", month)
          target <- as.integer(target == "BAJA+2")
        } else target <- rep(NA_integer_, nrow(frame))
        item <- list(signature = e$signature, month = month, data = frame, label = target)
        file <- paste0(month, ".rds"); h55_save(item, file.path(root, file))
        entries[[as.character(month)]] <- list(file = file, sha = digest::digest(file = file.path(root, file), algo = "sha256"), rows = nrow(frame))
        h55_save(list(signature = e$signature, month = month, raw = raw,
          entry = entries[[as.character(month)]]), file.path(root, paste0(month, ".commit.rds")))
        rm(frame, item); invisible(gc())
      }
      rm(repaired, labels, index); invisible(gc())
    }
    stopifnot(length(raw) == 152L, length(entries) == length(months))
    manifest <- list(signature = e$signature, phase = phase, train = train, evaluation = evaluation,
      raw = raw, entries = entries[as.character(months)])
    h55_save(manifest, file.path(root, "manifest.rds"))
  }
  stopifnot(identical(manifest$signature, e$signature), identical(manifest$train, train), manifest$evaluation == evaluation)
  for (entry in manifest$entries) {
    dest <- file.path(local, entry$file)
    if (!file.exists(dest) || !identical(digest::digest(file = dest, algo = "sha256"), entry$sha)) {
      source <- if (is.null(entry$durable_path)) file.path(root, entry$file) else entry$durable_path
      stopifnot(identical(digest::digest(file = source, algo = "sha256"), entry$sha))
      if (!file.copy(source, dest, overwrite = TRUE)) stop("Local staging failed")
    }
    stopifnot(identical(digest::digest(file = dest, algo = "sha256"), entry$sha))
  }
  e$manifest <- manifest; e$bank <- local
  e$views <- h55_views(manifest$raw)
  invisible(manifest)
}


## 11. Tareas nativas y recuperación de modelos


### `h55_task()`

Clave estable del ajuste. Identifica contrato, fase, columnas efectivas y parámetros completos, incluida la semilla. Dos nombres lógicos que representan exactamente el mismo ajuste pueden reutilizar el modelo; dos semillas diferentes no pueden confundirse.


In [ ]:
h55_task <- function(e, view, seed_index, overrides = list(), id = view, phase = e$manifest$phase) {
  columns <- e$views[[view]]; stopifnot(length(columns) >= 152L, !anyDuplicated(columns))
  seed <- e$seeds[seed_index]; params <- h55_params(seed, overrides, e$cfg$smoke)
  bank_sha <- h55_hash(lapply(e$manifest$entries, function(entry) entry[c("sha", "rows")]))
  key <- h55_hash(list(e$signature, bank_sha, phase, columns, params))
  list(id = id, key = key, view = view, columns = columns, seed_index = seed_index, seed = seed,
    params = params, phase = phase, bank_sha = bank_sha, forbid_fit = FALSE, interrupt_after_model = FALSE)
}


### `h55_frame()`

Carga de una vista mensual. ID, mes y etiqueta permanecen como claves o resultados, nunca como predictores. El orden de columnas y de clientes se conserva tanto al entrenar como al aplicar un modelo restaurado.


In [ ]:
h55_frame <- function(w, month, columns) {
  b <- readRDS(file.path(w$bank, paste0(month, ".rds")))
  stopifnot(identical(b$signature, w$signature), b$month == month,
    !any(c("numero_de_cliente", "foto_mes", "clase_ternaria") %in% columns))
  list(matrix = as.matrix(b$data[, columns, with = FALSE]), label = b$label,
    id = b$data$numero_de_cliente)
}


### `h55_fit()`

Una tarea nativa recuperable. Si ya hay resultado se verifica su modelo; si solo hay modelo confirmado, se restaura sin ajustar otra vez. El dataset binario tiene identidad de matriz/binning y bloqueo local. Tras confirmar los pesos durables se descartan los binarios de uso único; solo el dataset compartido de HPO se conserva durante esa etapa. Los pesos y resultados nunca se eliminan para ahorrar scratch.


In [ ]:
h55_fit <- function(w, task) {
  started <- as.numeric(Sys.time()); key <- task$key
  model_path <- file.path(w$cfg$output, "models", paste0(key, ".txt"))
  meta_path <- file.path(w$cfg$output, "models", paste0(key, ".rds"))
  result_path <- file.path(w$cfg$output, "results", paste0(key, ".rds"))
  meta <- h55_read(meta_path); result <- h55_read(result_path)
  if (!is.null(meta)) stopifnot(identical(meta$key, key), identical(meta$signature, w$signature),
    identical(meta$columns, task$columns), identical(meta$params, task$params),
    identical(meta$train_months, w$manifest$train), identical(meta$bank_sha, task$bank_sha),
    identical(digest::digest(file = model_path, algo = "sha256"), meta$model_sha))
  if (!is.null(result)) {
    stopifnot(!is.null(meta), identical(result$signature, w$signature), identical(result$key, key),
      identical(result$model_sha, meta$model_sha), result$seed == task$seed)
    return(list(metrics = result$metrics, fitted = FALSE, key = key, seconds = as.numeric(Sys.time()) - started))
  }
  fitted <- FALSE
  if (is.null(meta)) {
    if (task$forbid_fit) stop("Uncommitted model; refit forbidden: ", key)
    h55_disk(w$cfg)
    h55_log(w, "dataset_construction", task$id, key)
    bin_key <- h55_hash(list(w$signature, task$bank_sha, task$phase, task$columns, task$seed, task$params$max_bin,
      "bin_construct_sample_cnt=200000;feature_pre_filter=FALSE"))
    binary <- file.path(w$cfg$scratch, w$signature, "datasets", paste0(bin_key, ".bin"))
    dir.create(dirname(binary), recursive = TRUE, showWarnings = FALSE)
    guard <- paste0(binary, ".lock")
    while (!dir.create(guard, showWarnings = FALSE)) {
      owner <- h55_read(file.path(guard, "owner.rds"))
      if (!is.null(owner) && !dir.exists(paste0("/proc/", owner$pid))) unlink(guard, recursive = TRUE)
      else Sys.sleep(.1)
    }
    h55_save(list(pid = Sys.getpid()), file.path(guard, "owner.rds"))
    holding <- TRUE
    on.exit(if (holding) unlink(guard, recursive = TRUE), add = TRUE)
    dataset_params <- list(max_bin = task$params$max_bin, data_random_seed = task$seed,
      feature_pre_filter = FALSE, bin_construct_sample_cnt = 200000L, verbosity = -1L)
    binary_meta <- h55_read(paste0(binary, ".rds"))
    if (is.null(binary_meta)) {
      chunks <- lapply(w$manifest$train, function(m) h55_frame(w, m, task$columns))
      x <- do.call(rbind, lapply(chunks, `[[`, "matrix")); y <- unlist(lapply(chunks, `[[`, "label"), use.names = FALSE)
      rm(chunks); invisible(gc())
      dataset <- lightgbm::lgb.Dataset(x, label = y, params = dataset_params, free_raw_data = TRUE)
      tmp <- paste0(binary, ".uncommitted_", Sys.getpid())
      lightgbm::lgb.Dataset.save(dataset, tmp)
      if (!file.rename(tmp, binary)) stop("Binary dataset commit failed")
      h55_save(list(key = bin_key, sha = digest::digest(file = binary, algo = "sha256")), paste0(binary, ".rds"))
      rm(dataset, x, y); invisible(gc())
    }
    binary_meta <- readRDS(paste0(binary, ".rds"))
    stopifnot(identical(binary_meta$key, bin_key), identical(digest::digest(file = binary, algo = "sha256"), binary_meta$sha))
    unlink(guard, recursive = TRUE); holding <- FALSE
    dataset <- lightgbm::lgb.Dataset(binary, params = dataset_params)
    h55_log(w, "native_training", task$id, key)
    params <- task$params; rounds <- params$num_iterations; params$num_iterations <- NULL
    model <- lightgbm::lgb.train(params = params, data = dataset, nrounds = rounds)
    tmp <- tempfile(tmpdir = dirname(binary), fileext = ".model")
    lightgbm::lgb.save(model, tmp)
    dir.create(dirname(model_path), recursive = TRUE, showWarnings = FALSE)
    if (!file.copy(tmp, model_path, overwrite = TRUE)) stop("Durable model write failed")
    sha <- digest::digest(file = tmp, algo = "sha256")
    stopifnot(identical(digest::digest(file = model_path, algo = "sha256"), sha)); unlink(tmp)
    meta <- list(signature = w$signature, key = key, columns = task$columns, params = task$params,
      model_sha = sha, seed = task$seed, seed_index = task$seed_index,
      bank_sha = task$bank_sha,
      train_months = w$manifest$train, fit_seconds = as.numeric(Sys.time()) - started)
    h55_save(meta, meta_path); fitted <- TRUE
    h55_log(w, "model_commit", sha, key)
    rm(dataset, model); invisible(gc())
    if (!startsWith(task$id, "hpo_")) unlink(c(binary, paste0(binary, ".rds")))
    if (task$interrupt_after_model) stop("Injected interruption after durable model commit")
  }
  h55_log(w, "prediction", task$id, key)
  model <- lightgbm::lgb.load(model_path)
  evaluation <- h55_frame(w, w$manifest$evaluation, task$columns)
  prob <- as.numeric(predict(model, evaluation$matrix))
  stopifnot(length(prob) == length(evaluation$id), all(is.finite(prob)), all(prob >= 0 & prob <= 1))
  metrics <- if (task$phase == "final") NULL else h55_metrics(prob, evaluation$label, evaluation$id)
  result <- list(signature = w$signature, key = key, model_sha = meta$model_sha,
    metrics = metrics, predictions = data.table::data.table(numero_de_cliente = evaluation$id,
      foto_mes = w$manifest$evaluation, prob = prob), seed_index = task$seed_index, seed = task$seed)
  h55_save(result, result_path); h55_log(w, "metrics_commit", task$id, key)
  list(metrics = metrics, fitted = fitted, key = key, seconds = as.numeric(Sys.time()) - started)
}


### `h55_worker_job()`

Frontera de errores del trabajador. Devuelve únicamente métricas y metadatos pequeños, no matrices o punteros nativos. Registra PID, inicio, fin y VmHWM: ese pico corresponde a la vida del proceso y es una cota conservadora, no un pico aislado de cada ajuste. El controlador conserva los resultados exitosos aunque falle un compañero.


In [ ]:
h55_worker_job <- function(task) {
  started <- as.numeric(Sys.time())
  tryCatch({
    result <- h55_fit(h55_worker, task)
    status <- readLines("/proc/self/status"); peak <- status[startsWith(status, "VmHWM:")]
    list(success = TRUE, id = task$id, seed_index = task$seed_index, key = task$key, result = result,
      pid = Sys.getpid(), started = started, finished = as.numeric(Sys.time()),
      worker_lifetime_peak_gib = if (length(peak)) as.numeric(strsplit(trimws(sub(".*:", "", peak)), " +")[[1L]][1L]) / 1024^2 else NA_real_)
  }, error = function(err) list(success = FALSE, id = task$id, seed_index = task$seed_index,
    key = task$key, pid = Sys.getpid(), started = started, finished = as.numeric(Sys.time()), error = conditionMessage(err)))
}


## 12. Recursos y cola paralela


### `h55_resources()`

Memoria realmente disponible. Se consulta Linux en cada admisión en vez de asumir que toda la RAM nominal está libre. La medición es operativa y no modifica el contenido estadístico del estudio.


In [ ]:
h55_resources <- function() {
  mem <- readLines("/proc/meminfo")
  value <- function(name) as.numeric(strsplit(trimws(sub(".*:", "", mem[startsWith(mem, paste0(name, ":"))])), " +")[[1L]][1L]) / 1024^2
  list(total = value("MemTotal"), available = value("MemAvailable"))
}


### `h55_disk()`

Espacio y tipo de scratch. Se rechazan tmpfs y GCSFUSE: los datos activos deben estar en disco local. El cache binario acotado requiere 30 GiB libres al iniciar un estudio nuevo y mantiene una reserva operativa de 8 GiB. Se comprueba al preparar datos y antes de cada ajuste; nunca se borran checkpoints durables ni archivos de otros estudios para pasar el control.


In [ ]:
h55_disk <- function(cfg) {
  lines <- system2("df", c("-PTk", shQuote(cfg$scratch)), stdout = TRUE)
  parts <- strsplit(trimws(tail(lines, 1L)), " +")[[1L]]
  free <- as.double(parts[5L]) / 1024^2
  if (cfg$cloud && (parts[2L] %in% c("tmpfs", "ramfs", "devtmpfs") || grepl("gcsfuse", parts[2L], fixed = TRUE)))
    stop("Cloud scratch must use local disk, not RAM-backed or GCSFUSE storage")
  minimum <- if (file.exists(file.path(cfg$output, "contract.rds"))) cfg$disk_resume_min_gib else cfg$disk_min_gib
  if (!cfg$smoke && (!is.finite(free) || free < minimum))
    stop("Insufficient local staging disk: ", round(free, 1), " GiB; need ", minimum)
  invisible(free)
}


### `h55_estimate()`

Reserva de pico, no solo RSS actual. El cálculo promueve los conteos antes de multiplicar filas por columnas y agrega un margen para copias y buffers nativos. Es una estimación conservadora que debe contrastarse con las mediciones reales.


In [ ]:
h55_estimate <- function(rows, columns) {
  as.double(rows) * as.double(columns) * 8 / 1024^3 * 3 + 5
}


### `h55_admit()`

Admisión de una tarea nueva. Se reserva memoria para los ajustes que aún podrían crecer y para el sistema operativo. Tener CPU libre no obliga a iniciar un tercer ajuste si la memoria disponible no lo permite.


In [ ]:
h55_admit <- function(estimate, available, reserved, headroom) {
  stopifnot(all(is.finite(c(estimate, available, reserved, headroom))), estimate > 0)
  estimate + sum(reserved) + headroom <= available
}


### `h55_pool()`

Procesos persistentes sin sobresuscripción. Cada trabajador usa cuatro hilos LightGBM y uno de BLAS/data.table. Solo se exportan funciones, rutas y manifiestos; no se exportan el panel completo ni sockets del controlador.


In [ ]:
h55_pool <- function(e) {
  if (!is.null(e$cluster)) return(invisible(NULL))
  e$cluster <- parallel::makePSOCKcluster(e$cfg$workers, outfile = file.path(e$cfg$output, "workers.log"))
  parallel::clusterCall(e$cluster, function(paths) {
    .libPaths(paths); Sys.setenv(OMP_NUM_THREADS = 4L, OPENBLAS_NUM_THREADS = 1L, MKL_NUM_THREADS = 1L)
    library(data.table); data.table::setDTthreads(1L); NULL
  }, .libPaths())
  functions <- ls(envir = .GlobalEnv, pattern = "^h55_")
  functions <- functions[vapply(mget(functions, .GlobalEnv), is.function, logical(1))]
  parallel::clusterExport(e$cluster, functions, envir = .GlobalEnv)
  h55_worker <- list(cfg = e$cfg, signature = e$signature, bank = e$bank, manifest = e$manifest)
  parallel::clusterExport(e$cluster, "h55_worker", envir = environment())
  e$pids <- unlist(parallel::clusterCall(e$cluster, Sys.getpid))
  h55_save(list(host = Sys.info()[["nodename"]], controller = Sys.getpid(), workers = e$pids),
    file.path(e$cfg$output, "workers.rds"))
}


### `h55_pool_stop()`

Cierre antes de cambiar de etapa o liberar el bloqueo. Se comprueba que los procesos hayan desaparecido. Si no podemos confirmarlo, se conserva el bloqueo para impedir dos controladores entrenando en la misma salida.


In [ ]:
h55_pool_stop <- function(e) {
  if (!is.null(e$cluster)) { parallel::stopCluster(e$cluster); e$cluster <- NULL }
  for (i in 1:20) {
    if (!any(vapply(e$pids, function(pid) dir.exists(paste0("/proc/", pid)), logical(1)))) return(TRUE)
    Sys.sleep(.1)
  }
  !length(e$pids)
}


### `h55_queue()`

Cola con reposición al terminar. Los candidatos y sus semillas son independientes: un ajuste lento no impide ocupar un puesto libre con otra tarea. La admisión reserva picos futuros; al fallar una tarea se dejan terminar y guardar los pares activos sin iniciar trabajo nuevo.


In [ ]:
h55_queue <- function(e, tasks, stage, worker_fun = h55_worker_job, resource_probe = h55_resources) {
  if (!e$cfg$smoke && (!identical(worker_fun, h55_worker_job) || !identical(resource_probe, h55_resources)))
    stop("Custom worker/resource callbacks are fixture-only")
  if (!length(tasks)) return(list())
  keys <- vapply(tasks, `[[`, character(1), "key")
  if (anyDuplicated(keys)) stop("Duplicate model writers in queue")
  h55_pool(e)
  pending <- seq_along(tasks); active <- rep(NA_integer_, e$cfg$workers)
  reserved <- numeric(e$cfg$workers); results <- vector("list", length(tasks)); failed <- FALSE; deadline_hit <- FALSE
  rows <- sum(vapply(e$manifest$entries[as.character(e$manifest$train)], `[[`, numeric(1), "rows"))
  while (length(pending) || any(!is.na(active))) {
    if (!e$cfg$smoke && as.numeric(Sys.time()) - e$started >= (e$cfg$horizon_hours - e$cfg$recovery_hours) * 3600) {
      deadline_hit <- TRUE; failed <- TRUE
      if (!any(!is.na(active))) break
    }
    for (slot in which(is.na(active))) {
      if (!length(pending) || failed) break
      at <- pending[1L]; estimate <- if (e$cfg$smoke) .05 else h55_estimate(rows, length(tasks[[at]]$columns))
      free <- resource_probe()$available
      admit <- h55_admit(estimate, free, reserved, if (e$cfg$smoke) 0 else e$cfg$reserve_gib)
      h55_log(e, "memory_admission", paste(stage, tasks[[at]]$id, "allow", admit, "available", round(free, 2), "reserved", round(sum(reserved), 2), "next", round(estimate, 2)))
      if (!admit) {
        e$admission_cap <- min(e$admission_cap, sum(!is.na(active)))
        break
      }
      parallel:::sendCall(e$cluster[[slot]], worker_fun, list(tasks[[at]]), tag = at)
      active[slot] <- at; reserved[slot] <- estimate; pending <- pending[-1L]
    }
    if (!any(!is.na(active))) {
      if (failed) break
      stop("Insufficient available RAM for next fit; no task started")
    }
    received <- parallel:::recvOneResult(e$cluster)
    slot <- received$node; at <- active[slot]; result <- received$value
    if (!is.list(result) || is.null(result$success)) stop("Worker transport failure; inspect worker PIDs before lock takeover")
    results[[at]] <- result; active[slot] <- NA_integer_; reserved[slot] <- 0
    result$stage <- stage; e$ledger[[length(e$ledger) + 1L]] <- result
    h55_save(e$ledger, file.path(e$cfg$output, "jobs.rds"))
    failed <- failed || !result$success
    h55_log(e, "queue_progress", paste(stage, sum(!vapply(results, is.null, logical(1))), "/", length(tasks)))
    if (failed && !any(!is.na(active))) break
  }
  if (deadline_hit) stop("Time reserve reached: active peers committed; no new fits admitted")
  if (failed) stop("Worker failure; successful peers preserved: ", paste(vapply(Filter(function(x) !is.null(x) && !x$success, results), `[[`, character(1), "error"), collapse = " | "))
  results
}


### `h55_forecast()`

Admisión temporal basada en mediciones contadas. Las primeras tres semillas del baseline son parte de sus cinco ajustes previstos, no benchmarks adicionales. La proyección incluye HPO y producción y utiliza el número conservador de puestos admitidos. Si no cabe en las 48 horas restantes, se pausa para revisión; no se mata un ajuste a mitad de entrenamiento.


In [ ]:
h55_forecast <- function(e, stage, feature_pending = 0L) {
  if (e$cfg$smoke) return(invisible(NULL))
  fit <- Filter(function(x) isTRUE(x$success) && isTRUE(x$result$fitted), e$ledger)
  if (!length(fit)) stop("No counted fit timings available for admission")
  seconds <- vapply(fit, function(x) x$finished - x$started, numeric(1))
  slow <- max(seconds) * 1.5
  slots <- max(1L, min(e$cfg$workers, e$admission_cap))
  hpo_waves <- ceiling(3 / slots) + ceiling(5 / slots) + 4 * ceiling(3 / slots)
  forecast <- switch(stage,
    feature = ceiling(feature_pending / slots) * slow + ceiling(5 / slots) * slow + hpo_waves * slow * 3,
    hpo = hpo_waves * slow,
    final = ceiling(5 / slots) * slow * 1.25,
    stop("Unknown timing phase"))
  remaining <- e$cfg$horizon_hours * 3600 - (as.numeric(Sys.time()) - e$started)
  needed <- forecast + (e$cfg$final_hours + e$cfg$recovery_hours) * 3600
  h55_log(e, "time_admission", paste(stage, "estimated_seconds", round(needed), "remaining_seconds", round(remaining), "slots", slots))
  if (needed > remaining) stop("Workload does not fit remaining 48-hour envelope; pause for an explicit budget change")
}


## 13. Comparaciones y selección de paquetes


### `h55_contrasts()`

Comparaciones pareadas por índice de semilla. Se guardan las cinco AUC y ganancias, sus diferencias contra el baseline y resumen con media, desviación, mínimo y máximo de AUC. No elegimos la mejor semilla ni presentamos las cinco réplicas como cinco períodos independientes.


In [ ]:
h55_contrasts <- function(e, tasks) {
  rows <- data.table::rbindlist(lapply(tasks, function(t) {
    result <- readRDS(file.path(e$cfg$output, "results", paste0(t$key, ".rds")))
    data.table::as.data.table(c(list(view = t$view, seed_index = t$seed_index, seed = t$seed, key = t$key), result$metrics))
  }))
  baseline <- rows[view == "snapshot152_repaired", .(seed_index, baseline_auc = auc, baseline_gain7 = gain7)]
  rows <- merge(rows, baseline, by = "seed_index", sort = FALSE)
  rows[, `:=`(delta_auc = auc - baseline_auc, delta_gain7 = gain7 - baseline_gain7)]
  summary <- rows[, .(mean_auc = mean(auc), sd_auc = sd(auc), min_auc = min(auc), max_auc = max(auc),
    mean_delta_auc = mean(delta_auc), sd_delta_auc = sd(delta_auc), seeds_improved = sum(delta_auc > 0),
    mean_gain7 = mean(gain7), sd_gain7 = sd(gain7), min_gain7 = min(gain7), max_gain7 = max(gain7),
    mean_delta_gain7 = mean(delta_gain7), sd_delta_gain7 = sd(delta_gain7),
    mean_logloss = mean(logloss), sd_logloss = sd(logloss), min_logloss = min(logloss), max_logloss = max(logloss),
    mean_brier = mean(brier), sd_brier = sd(brier), min_brier = min(brier), max_brier = max(brier)), by = view]
  list(per_seed = rows, summary = summary)
}


### `h55_select()`

Regla de selección confirmada antes de mirar mayo. Se incorpora un paquete si su diferencia media de AUC es positiva y mejora en al menos cuatro de cinco semillas. El baseline es la referencia, no un paquete candidato.


In [ ]:
h55_select <- function(summary) {
  allowed <- c("lag1_core", "delta1_core", "prior3_core", "relative3_funds", "card_utilization")
  summary[view %in% allowed & mean_delta_auc > 0 & seeds_improved >= 4L, view]
}


## 14. HPO bayesiana por lotes


### `h55_decode()`

Coordenadas del optimizador a parámetros nativos. Rondas, hojas y soporte mínimo se redondean; learning rate y soporte usan escala logarítmica. Se deduplican los parámetros efectivos, no solo los puntos continuos antes del redondeo.


In [ ]:
h55_decode <- function(x) list(num_iterations = as.integer(round(64 + 320 * x[1L])),
  learning_rate = exp(log(.02) + x[2L] * log(5)), num_leaves = as.integer(round(16 + 48 * x[3L])),
  min_data_in_leaf = as.integer(round(exp(log(300) + x[4L] * log(5)))), feature_fraction = .5 + .5 * x[5L])


### `h55_encode()`

Representación normalizada para el proceso gaussiano. Convierte parámetros efectivos a cinco coordenadas comparables. Esta normalización es del optimizador, no de las variables bancarias ni de sus etiquetas.


In [ ]:
h55_encode <- function(p) c((p$num_iterations - 64) / 320, log(p$learning_rate / .02) / log(5),
  (p$num_leaves - 16) / 48, log(p$min_data_in_leaf / 300) / log(5), (p$feature_fraction - .5) / .5)


### `h55_propose()`

Bayesiana por lotes con Kriging believer. Uno de los ocho puntos iniciales es exactamente la sonda ya ajustada, sin redondeos numéricos que impidan reutilizarla. Después el surrogate propone tres configuraciones distintas mediante fantasías de media posterior y mejora esperada. Las propuestas tienen RNG reproducible y no dependen del orden de finalización.


In [ ]:
h55_propose <- function(observed, batch, master = 333803L) {
  h55_rng(as.integer(master + 1000L + batch), function() {
    if (batch == 0L) {
      points <- rbind(c(1, 1, 1, 0, 1), h55_encode(h55_params(1L)), matrix(runif(30L), ncol = 5L))
      params <- lapply(seq_len(nrow(points)), function(i) h55_decode(points[i, ]))
      anchor <- h55_params(1L)
      params[[2L]] <- anchor[c("num_iterations", "learning_rate", "num_leaves", "min_data_in_leaf", "feature_fraction")]
      stopifnot(!anyDuplicated(vapply(params, h55_hash, character(1))))
      return(list(params = params, observed = observed, proposal_seed = master + 1000L, fantasies = list()))
    }
    pool <- lapply(seq_len(2048L), function(i) h55_decode(runif(5L)))
    seen <- vapply(observed$params, h55_hash, character(1))
    pool <- pool[!duplicated(vapply(pool, h55_hash, character(1)))]
    pool <- pool[!vapply(pool, h55_hash, character(1)) %in% seen]
    x <- do.call(rbind, lapply(observed$params, h55_encode)); y <- observed$auc
    center <- mean(y); scale <- sd(y)
    if (!is.finite(scale) || scale < 1e-12) stop("HPO observations flat: stop for review rather than claiming adaptive optimization")
    y <- (y - center) / scale; proposed <- list(); fantasies <- list()
    for (k in 1:3) {
      design <- as.data.frame(x); names(design) <- paste0("x", 1:5)
      model <- DiceKriging::km(design = design, response = y, covtype = "matern5_2", nugget = 1e-8,
        nugget.estim = FALSE, lower = rep(.05, 5L), upper = rep(2, 5L), control = list(trace = FALSE))
      candidates <- as.data.frame(do.call(rbind, lapply(pool, h55_encode))); names(candidates) <- names(design)
      pred <- predict(model, candidates, type = "UK", se.compute = TRUE, cov.compute = FALSE)
      s <- pmax(as.numeric(pred$sd), 1e-12); improvement <- as.numeric(pred$mean) - max(y)
      z <- improvement / s; ei <- improvement * pnorm(z) + s * dnorm(z)
      at <- which.max(ei); proposed[[k]] <- pool[[at]]
      fantasies[[k]] <- list(posterior_mean = pred$mean[at] * scale + center, expected_improvement = ei[at])
      x <- rbind(x, as.numeric(candidates[at, ])); y <- c(y, pred$mean[at]); pool <- pool[-at]
    }
    list(params = proposed, observed = observed, proposal_seed = master + 1000L + batch, fantasies = fantasies)
  })
}


### `h55_hpo()`

Una sola HPO sobre columnas congeladas. Ocho puntos iniciales y cuatro lotes de tres dan veinte evaluaciones, pero solo hasta diecinueve ajustes nuevos: el punto 2 reutiliza la sonda de confirmación con FIT_SEEDS[1]. Las propuestas se guardan antes de ejecutar y se incorporan por ID canónico. Los evaluadores simulados solo están permitidos en pruebas sintéticas.


In [ ]:
h55_hpo <- function(e, execute = h55_queue) {
  if (!e$cfg$smoke && !identical(execute, h55_queue)) stop("Custom HPO evaluators are fixture-only")
  observed <- list(params = list(), auc = numeric())
  all_tasks <- list()
  for (batch in 0:4) {
    path <- file.path(e$cfg$output, "hpo", paste0("batch_", batch, ".rds"))
    proposal <- h55_read(path)
    if (is.null(proposal)) {
      proposal <- h55_propose(observed, batch, e$cfg$master)
      proposal$signature <- e$signature; proposal$view_hash <- h55_hash(e$views$aggregate)
      proposal$seed <- e$seeds[1L]; h55_save(proposal, path)
    }
    stopifnot(identical(proposal$signature, e$signature), identical(proposal$observed, observed),
      identical(proposal$view_hash, h55_hash(e$views$aggregate)), proposal$seed == e$seeds[1L])
    tasks <- lapply(seq_along(proposal$params), function(i) h55_task(e, "aggregate", 1L,
      proposal$params[[i]], id = paste0("hpo_", batch, "_", i)))
    if (batch == 0L) {
      if (!e$cfg$smoke) {
        anchor <- h55_task(e, "aggregate", 1L)
        stopifnot(identical(tasks[[2L]]$key, anchor$key),
          file.exists(file.path(e$cfg$output, "results", paste0(anchor$key, ".rds"))))
      }
      execute(e, tasks[1:3], "hpo_counted_timing_batch")
      h55_forecast(e, "hpo")
      execute(e, tasks[-(1:3)], "hpo_initial")
    } else execute(e, tasks, paste0("hpo_batch_", batch))
    for (i in seq_along(tasks)) {
      result <- readRDS(file.path(e$cfg$output, "results", paste0(tasks[[i]]$key, ".rds")))
      observed$params[[length(observed$params) + 1L]] <- proposal$params[[i]]
      observed$auc <- c(observed$auc, result$metrics$auc)
    }
    all_tasks <- c(all_tasks, tasks)
    h55_save(observed, file.path(e$cfg$output, "hpo", "observations.rds"))
  }
  best <- which.max(observed$auc)
  winner <- list(signature = e$signature, params = observed$params[[best]], auc = observed$auc[best],
    trial = best, key = all_tasks[[best]]$key, tie_rule = "earliest canonical trial on equal AUC")
  h55_save(winner, file.path(e$cfg$output, "hpo", "winner.rds")); winner
}


## 15. Entregas y protección de Kaggle


### `h55_submissions()`

Archivos de septiembre deterministas. La predicción final es la media de las cinco probabilidades, no la media de sus rankings. Se generan nueve cortes entre 8000 y 12000, en pasos de 500, con desempate por ID y el esquema numero_de_cliente/Predicted.


In [ ]:
h55_submissions <- function(predictions, cutoffs, output) {
  stopifnot(!anyDuplicated(predictions$numero_de_cliente), all(is.finite(predictions$prob)),
    all(cutoffs > 0), all(cutoffs <= nrow(predictions)))
  ranked <- order(-predictions$prob, predictions$numero_de_cliente); files <- list()
  for (n in cutoffs) {
    table <- data.table::data.table(numero_de_cliente = predictions$numero_de_cliente, Predicted = 0L)
    data.table::set(table, i = ranked[seq_len(n)], j = "Predicted", value = 1L)
    data.table::setorder(table, numero_de_cliente)
    path <- file.path(output, "submissions", paste0("HT5955_contacts_", n, ".csv"))
    dir.create(dirname(path), recursive = TRUE, showWarnings = FALSE)
    data.table::fwrite(table, path)
    files[[as.character(n)]] <- list(path = path, contacts = n, sha = digest::digest(file = path, algo = "sha256"))
  }
  files
}


### `h55_upload()`

Subida explícitamente autorizada y registrada. El usuario habilitó SEND_KAGGLE y confirmó la competencia en los controles de esta entrega; el modo sintético siempre desactiva las subidas. Cada archivo tiene SHA y recibo; una solicitud pendiente o ambigua detiene los reintentos para evitar duplicar una entrega. No se consulta el leaderboard para ajustar el estudio.


In [ ]:
h55_upload <- function(cfg, files) {
  if (!cfg$send_kaggle) return(invisible("Uploads OFF"))
  if (cfg$smoke || !cfg$cloud || !cfg$competition_confirmed || !nzchar(Sys.which("kaggle")))
    stop("Upload requires full cloud mode, confirmed competition and Kaggle CLI")
  for (file in files) {
    stopifnot(identical(digest::digest(file = file$path, algo = "sha256"), file$sha))
    path <- file.path(cfg$output, "uploads", paste0(file$sha, ".rds")); old <- h55_read(path)
    if (!is.null(old)) {
      if (identical(old$status, "confirmed_success")) next
      stop("Pending/ambiguous submission: inspect saved CLI response and confirm receipt manually; no automatic retry")
    }
    h55_save(list(status = "pending", competition = cfg$competition, file = file, requested = as.character(Sys.time())), path)
    response <- suppressWarnings(system2("kaggle", c("competitions", "submit", "-c", shQuote(cfg$competition),
      "-f", shQuote(file$path), "-m", shQuote(paste0("HT5955 fixed recipe, contacts=", file$contacts))), stdout = TRUE, stderr = TRUE))
    status <- attr(response, "status"); if (is.null(status)) status <- 0L
    confirmed <- status == 0L && any(grepl("Successfully submitted", response, ignore.case = TRUE))
    h55_save(list(status = if (confirmed) "confirmed_success" else "ambiguous", competition = cfg$competition,
      file = file, response = response, exit_status = status), path)
    if (!confirmed) stop("Submission not confirmed; no automatic retry or leaderboard query")
  }
  invisible(NULL)
}


## 16. Orquestador del experimento


### `h55_run()`

Orquestación de extremo a extremo. Desarrollo compara cinco paquetes contra un solo baseline de 152 predictores; la combinación seleccionada debe confirmar la misma regla de cuatro semillas. Si falla, se detiene sin búsqueda de subconjuntos. Después se congela la receta, se hace HPO, se entrena hasta julio y se aplica a septiembre. El smoke ejecuta exactamente cinco ajustes diminutos y no entra a HPO o Kaggle.


In [ ]:
h55_run <- function(cfg, plan, fixture = NULL, forbid_fit = FALSE) {
  e <- h55_setup(cfg, plan, if (!is.null(fixture)) h55_hash(as.data.frame(fixture)) else NULL)
  lock <- e$lock
  on.exit({
    dead <- tryCatch(h55_pool_stop(e), error = function(err) FALSE)
    if (dead) h55_unlock(lock) else warning("Workers may still be alive; retaining run lock")
  }, add = TRUE)
  finished <- h55_read(file.path(cfg$output, "september_ensemble.rds"))
  if (!cfg$smoke && !is.null(finished)) {
    stopifnot(identical(finished$signature, e$signature))
    files <- h55_read(file.path(cfg$output, "submission_manifest.rds"))
    if (is.null(files)) {
      files <- h55_submissions(finished$predictions, cfg$cutoffs, cfg$output)
      h55_save(files, file.path(cfg$output, "submission_manifest.rds"))
    }
    for (file in files) stopifnot(identical(digest::digest(file = file$path, algo = "sha256"), file$sha))
    h55_upload(cfg, files)
    return(list(comparison = readRDS(file.path(cfg$output, "feature_comparison.rds"))$summary,
      selected = readRDS(file.path(cfg$output, "feature_freeze.rds"))$packages,
      hpo = finished$winner, submissions = files))
  }
  h55_prepare(e, fixture = fixture)
  if (cfg$smoke) {
    tasks <- lapply(1:5, function(i) h55_task(e, "lag1_core", i))
    tasks <- lapply(tasks, function(t) {t$forbid_fit <- forbid_fit; t})
    h55_queue(e, tasks, "smoke_exactly_five_fits")
    return(list(signature = e$signature, tasks = tasks,
      metrics = lapply(tasks, function(t) readRDS(file.path(cfg$output, "results", paste0(t$key, ".rds")))$metrics)))
  }
  views <- names(e$views)
  tasks <- unlist(lapply(views, function(view) lapply(1:5, function(i) h55_task(e, view, i))), recursive = FALSE)
  pilot <- 1:3
  h55_queue(e, tasks[pilot], "counted_controls_timing")
  h55_forecast(e, "feature", length(tasks) - length(pilot))
  h55_queue(e, tasks[-pilot], "feature_candidates")
  comparison <- h55_contrasts(e, tasks); h55_save(comparison, file.path(cfg$output, "feature_comparison.rds"))
  selected <- h55_select(comparison$summary)
  e$views$aggregate <- unique(c(e$manifest$raw, unlist(lapply(selected, function(v) setdiff(e$views[[v]], e$manifest$raw)))))
  aggregate <- lapply(1:5, function(i) h55_task(e, "aggregate", i))
  h55_queue(e, aggregate, "aggregate_confirmation")
  confirmation <- h55_contrasts(e, c(tasks[1:5], aggregate))
  row <- confirmation$summary[view == "aggregate"]
  if (length(selected) && !(row$mean_delta_auc > 0 && row$seeds_improved >= 4L))
    stop("Aggregate failed predeclared confirmation; stop for review, no subset search or HPO")
  freeze <- list(signature = e$signature, packages = selected, columns = e$views$aggregate,
    comparison = comparison$summary, confirmation = confirmation)
  path <- file.path(cfg$output, "feature_freeze.rds"); old <- h55_read(path)
  if (!is.null(old) && !identical(h55_hash(old), h55_hash(freeze))) stop("Frozen feature recipe mismatch")
  if (is.null(old)) h55_save(freeze, path)
  winner <- h55_hpo(e)
  h55_forecast(e, "final")
  if (!h55_pool_stop(e)) stop("Workers not confirmed stopped before final preparation")
  datasets <- file.path(cfg$scratch, e$signature, "datasets")
  unlink(list.files(datasets, pattern = "[.]bin([.]rds)?$", full.names = TRUE))
  h55_prepare(e, "final"); e$views$aggregate <- freeze$columns
  final <- lapply(1:5, function(i) h55_task(e, "aggregate", i, winner$params, id = paste0("final_", i)))
  h55_queue(e, final, "final_training")
  predictions <- lapply(final, function(t) readRDS(file.path(cfg$output, "results", paste0(t$key, ".rds")))$predictions)
  stopifnot(all(vapply(predictions, function(p) identical(p$numero_de_cliente, predictions[[1L]]$numero_de_cliente), logical(1))))
  average <- data.table::copy(predictions[[1L]]); average[, prob := Reduce(`+`, lapply(predictions, `[[`, "prob")) / 5]
  h55_save(list(signature = e$signature, feature_freeze_sha = digest::digest(file = path, algo = "sha256"),
    winner = winner, predictions = average, seed_keys = vapply(final, `[[`, character(1), "key")), file.path(cfg$output, "september_ensemble.rds"))
  files <- h55_submissions(average, cfg$cutoffs, cfg$output); h55_save(files, file.path(cfg$output, "submission_manifest.rds"))
  h55_upload(cfg, files)
  list(comparison = comparison$summary, selected = selected, hpo = winner, submissions = files)
}


## 17. Configurar y revisar antes de iniciar

Las rutas siguientes son absolutas en la VM. La salida nueva no toca HT5954 ni
sus bloqueos. El scratch está en `/content`, no en `/tmp`. El valor de WORKERS
puede bajar por recursos sin cambiar semillas ni paquetes; no puede superar tres.

La tabla de semillas es informativa: sus cinco filas son réplicas estocásticas,
no meses ni divisiones de clientes. El control de recursos exige una confirmación
humana para evitar consumir toda la máquina mientras otro estudio sigue activo.


In [ ]:
cfg <- h55_config(cloud = CLOUD_MODE, smoke = SAMPLE_MODE)
cfg$master <- as.integer(plan$master_seed)
cfg$workers <- WORKERS
cfg$full_resources_confirmed <- FULL_RESOURCES_CONFIRMED
cfg$send_kaggle <- SEND_KAGGLE
cfg$competition <- COMPETITION
cfg$competition_confirmed <- COMPETITION_CONFIRMED
cfg$resume_dead_token <- RESUME_DEAD_RUN_TOKEN
cfg$dead_workers_confirmed <- DEAD_WORKERS_CONFIRMED
data.frame(indice = 1:5, semilla = h55_seeds(cfg$master))
cfg[c("input", "output", "scratch", "workers", "threads", "send_kaggle")]


### Esquema de referencia para la prueba sintética

Los 152 nombres provienen del diccionario reconciliado del directorio. Esta lista
solo construye el fixture técnico cuando SAMPLE_MODE es TRUE; la corrida real
verifica el esquema del CSV y su SHA256. El fixture no contiene datos personales.


In [ ]:
RAW_FIELDS <- c(
  "active_quarter", "cliente_vip", "internet", "cliente_edad",
  "cliente_antiguedad", "mrentabilidad", "mrentabilidad_annual", "mcomisiones",
  "mactivos_margen", "mpasivos_margen", "cproductos", "tcuentas",
  "ccuenta_corriente", "mcuenta_corriente_adicional", "mcuenta_corriente", "ccaja_ahorro",
  "mcaja_ahorro", "mcaja_ahorro_adicional", "mcaja_ahorro_dolares", "cdescubierto_preacordado",
  "mcuentas_saldo", "ctarjeta_debito", "ctarjeta_debito_transacciones", "mautoservicio",
  "ctarjeta_visa", "ctarjeta_visa_transacciones", "mtarjeta_visa_consumo", "ctarjeta_master",
  "ctarjeta_master_transacciones", "mtarjeta_master_consumo", "cprestamos_personales", "mprestamos_personales",
  "cprestamos_prendarios", "mprestamos_prendarios", "cprestamos_hipotecarios", "mprestamos_hipotecarios",
  "cplazo_fijo", "mplazo_fijo_dolares", "mplazo_fijo_pesos", "cinversion1",
  "minversion1_pesos", "minversion1_dolares", "cinversion2", "minversion2",
  "cseguro_vida", "cseguro_auto", "cseguro_vivienda", "cseguro_accidentes_personales",
  "ccaja_seguridad", "cpayroll_trx", "mpayroll", "mpayroll2",
  "cpayroll2_trx", "ccuenta_debitos_automaticos", "mcuenta_debitos_automaticos", "ctarjeta_visa_debitos_automaticos",
  "mttarjeta_visa_debitos_automaticos", "ctarjeta_master_debitos_automaticos", "mttarjeta_master_debitos_automaticos", "cpagodeservicios",
  "mpagodeservicios", "cpagomiscuentas", "mpagomiscuentas", "ccajeros_propios_descuentos",
  "mcajeros_propios_descuentos", "ctarjeta_visa_descuentos", "mtarjeta_visa_descuentos", "ctarjeta_master_descuentos",
  "mtarjeta_master_descuentos", "ccomisiones_mantenimiento", "mcomisiones_mantenimiento", "ccomisiones_otras",
  "mcomisiones_otras", "cforex", "cforex_buy", "mforex_buy",
  "cforex_sell", "mforex_sell", "ctransferencias_recibidas", "mtransferencias_recibidas",
  "ctransferencias_emitidas", "mtransferencias_emitidas", "cextraccion_autoservicio", "mextraccion_autoservicio",
  "ccheques_depositados", "mcheques_depositados", "ccheques_emitidos", "mcheques_emitidos",
  "ccheques_depositados_rechazados", "mcheques_depositados_rechazados", "ccheques_emitidos_rechazados", "mcheques_emitidos_rechazados",
  "tcallcenter", "ccallcenter_transacciones", "thomebanking", "chomebanking_transacciones",
  "ccajas_transacciones", "ccajas_consultas", "ccajas_depositos", "ccajas_extracciones",
  "ccajas_otras", "catm_trx", "matm", "catm_trx_other",
  "matm_other", "ctrx_quarter", "tmobile_app", "cmobile_app_trx",
  "Master_delinquency", "Master_status", "Master_mfinanciacion_limite", "Master_Fvencimiento",
  "Master_Finiciomora", "Master_msaldototal", "Master_msaldopesos", "Master_msaldodolares",
  "Master_mconsumospesos", "Master_mconsumosdolares", "Master_mlimitecompra", "Master_madelantopesos",
  "Master_madelantodolares", "Master_fultimo_cierre", "Master_mpagado", "Master_mpagospesos",
  "Master_mpagosdolares", "Master_fechaalta", "Master_mconsumototal", "Master_cconsumos",
  "Master_cadelantosefectivo", "Master_mpagominimo", "Visa_delinquency", "Visa_status",
  "Visa_mfinanciacion_limite", "Visa_Fvencimiento", "Visa_Finiciomora", "Visa_msaldototal",
  "Visa_msaldopesos", "Visa_msaldodolares", "Visa_mconsumospesos", "Visa_mconsumosdolares",
  "Visa_mlimitecompra", "Visa_madelantopesos", "Visa_madelantodolares", "Visa_fultimo_cierre",
  "Visa_mpagado", "Visa_mpagospesos", "Visa_mpagosdolares", "Visa_fechaalta",
  "Visa_mconsumototal", "Visa_cconsumos", "Visa_cadelantosefectivo", "Visa_mpagominimo"
)


## 18. Ejecución explícita

Para la corrida real, abrir esta versión actualizada, reiniciar su kernel y
ejecutar desde el principio. `RUN_FULL` y `FULL_RESOURCES_CONFIRMED` ya están en
TRUE; CLOUD_MODE está en TRUE y SAMPLE_MODE en FALSE. `SEND_KAGGLE` y la confirmación
de competencia ya están en TRUE por autorización del usuario. Cambiar
`SEND_KAGGLE <- FALSE` si se desea revisar los nueve CSV antes de subirlos. No hay consultas al
leaderboard ni reconfiguración de la VM.

La prueba sintética usa una salida separada bajo `/tmp/opencode` SOLO en local,
o bajo `/content` en nube. Son exactamente cinco modelos de cinco rondas y una
sola vista; no ejecuta una mini búsqueda de candidatos ni una HPO reducida.


In [ ]:
result <- NULL
if (SAMPLE_MODE) {
  cfg$cloud <- FALSE
  cfg$send_kaggle <- FALSE
  parent <- if (dir.exists("/content")) "/content" else "/tmp/opencode"
  cfg$output <- file.path(parent, "ht5955_notebook_smoke")
  cfg$scratch <- file.path(parent, "ht5955_notebook_smoke_scratch")
  result <- h55_run(cfg, plan, fixture = h55_fixture(RAW_FIELDS))
} else if (RUN_FULL) {
  result <- h55_run(cfg, plan)
} else {
  message("Entrenamiento desactivado. Revisar el plan y habilitar RUN_FULL explícitamente.")
}


## 19. Cómo interpretar los resultados

Consultar `feature_comparison.rds`: todas las semillas, paquetes, controles y
diferencias pareadas, no solo el ganador. Un valor positivo en una semilla no
basta. `feature_freeze.rds` documenta qué paquetes se unieron y la confirmación.
`hpo/observations.rds` conserva las veinte pruebas y `winner.rds` sus parámetros.
Una desviación entre semillas no es un intervalo de generalización temporal.

El modelo aprende BAJA+2; la ganancia secundaria es
`1000000 * verdaderos_positivos - 25000 * contactos`. AUC y ganancia no son la
misma función: una mejor AUC puede no mejorar un corte concreto. El 7% es un
diagnóstico fijo, no una cuota de negocio ni el corte optimizado en mayo.

En septiembre se promedian probabilidades de los cinco modelos finales. Los
nueve archivos de 8000–12000 contactos son la salida solicitada del flujo senior,
no nueve oportunidades para adaptar el estudio al leaderboard. Para subirlos hay
que habilitar y confirmar la competencia; se conservan respuestas y hashes. Una
respuesta ambigua exige revisión manual, nunca un reintento ciego.


In [ ]:
if (!is.null(result)) {
  if (SAMPLE_MODE) {
    print(data.table::rbindlist(result$metrics))
    message("Prueba técnica sintética: estas métricas NO son resultados del estudio.")
  } else {
    print(result$comparison)
    print(result$selected)
    print(result$hpo[c("trial", "auc", "params")])
    print(vapply(result$submissions, function(x) x$contacts, numeric(1)))
  }
}


## 20. Recuperación, límites y comprobaciones

### Después de una interrupción Spot

1. Verificar que el controlador anterior **y todos sus trabajadores** terminaron.
2. Conservar este notebook, el mismo plan, fuente y versiones nativas. No borrar
   contratos, modelos confirmados o propuestas HPO para forzar compatibilidad.
3. Si quedó un bloqueo, leer su token y los PID registrados en `workers.rds`.
   Solo tras verificar la muerte de los procesos completar RESUME_DEAD_RUN_TOKEN
   y DEAD_WORKERS_CONFIRMED. El token no debe pegarse dentro de funciones.
4. Reiniciar desde el principio. El staging local es reconstruible y los shards
   durables se verifican. Un modelo confirmado recupera predicciones sin refit;
   un ajuste nativo que murió antes del commit debe comenzar de nuevo.
5. Las propuestas HPO pendientes mantienen sus parámetros y semillas; no se
   vuelve a sortear un lote porque su trabajador haya sido interrumpido.

### Evidencia y límites de validación

La validación local separa los checks aritméticos, fórmulas, temporalidad,
reparaciones, planificación de memoria y propuestas bayesianas sintéticas de la
prueba nativa de cinco ajustes. Una segunda ejecución cache-only verifica que no
se reentrene. El notebook limpio no guarda outputs; la copia ejecutada local se
mantiene separada. Estas pruebas no sustituyen medir CPU, disco y RAM en la VM.

No se promete un tiempo exacto sin medición. Se respetan los recursos de HT5954,
no se modifican daemons de inactividad y no se instalan keepalives. Si faltan
recursos o el presupuesto no alcanza, se informa y se pausa antes de cambiar el
experimento. Un fallo de confirmación de variables tampoco autoriza ampliar la
búsqueda.

### Fuentes metodológicas

Diccionario oficial reconciliado y `DATA_SEMANTICS_AND_AVAILABILITY.md` para
unidades, estados y disponibilidad; `MODELING_FOUNDATIONS.md` para relaciones
entre árboles, transformaciones y selección; `z635_WorkFlow_01_senior.ipynb` como
referencia de etapas, no como implementación causal verificada. Documentación
LightGBM: [valores faltantes](https://lightgbm.readthedocs.io/en/stable/Advanced-Topics.html),
[parámetros](https://lightgbm.readthedocs.io/en/stable/Parameters.html) y
[crecimiento y regularización](https://lightgbm.readthedocs.io/en/stable/Parameters-Tuning.html).
